## NB_lab_test_trends (Silver to Gold)
* **Target:** `<catalog>.gold.lab_test_trends`, one row per test type (`test_name`) per calendar month
* **Driving table:** `<catalog>.silver.lab_results`, grouped by `test_name`, year and month of `test_date`
* **Joins (LEFT):** `calendar` on the first day of the month, for `month_name` and `quarter_name`
* **Window:** `LAG(abnormal_rate_pct)` partitioned by `test_name`, ordered by `year`, `month` (month-over-month trend)
* **Load type:** FULL rebuild with `CREATE OR REPLACE TABLE ... AS SELECT`

Implementation notes (based on the silver schema):
* The calendar key is `datekey` (INT, yyyyMMdd). The first of the month is `year*10000 + month*100 + 1`, which matches `date_format(date_trunc('month', test_date), 'yyyyMMdd')`.
* `lab_results` contains duplicate `lab_result_id` rows from repeated loads. It is deduplicated to the latest row per key before aggregating.
* `abnormal_rate_change_pct` is the difference in percentage points between the current and previous month's rates (rounded rates).

In [0]:
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")
catalog = dbutils.widgets.get("catalog")

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.lab_test_trends"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gold")
print(f"Building {target_table} from {source_schema}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table}
COMMENT 'Gold: one row per lab test type per month with outcome rates and month-over-month trend'
AS
WITH lab AS (
  SELECT *
  FROM {source_schema}.lab_results
  QUALIFY ROW_NUMBER() OVER (PARTITION BY lab_result_id
                             ORDER BY update_timestamp DESC, landing_timestamp DESC) = 1
),
monthly AS (
  SELECT
    test_name,
    YEAR(test_date)                        AS year,
    MONTH(test_date)                       AS month,
    COUNT(*)                               AS total_tests,
    COUNT_IF(result_status = 'Abnormal')   AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')   AS critical_test_count
  FROM lab
  GROUP BY test_name, YEAR(test_date), MONTH(test_date)
),
rates AS (
  SELECT
    m.*,
    CAST(ROUND(abnormal_test_count / total_tests * 100, 2) AS DECIMAL(5,2)) AS abnormal_rate_pct,
    CAST(ROUND(critical_test_count / total_tests * 100, 2) AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly m
),
with_lag AS (
  SELECT
    r.*,
    LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates r
)
SELECT
  w.test_name,
  CAST(w.year AS INT)                                                     AS year,
  CAST(w.month AS INT)                                                    AS month,
  c.month_name,
  c.quarter_name,
  CAST(w.total_tests AS INT)                                              AS total_tests,
  CAST(w.abnormal_test_count AS INT)                                      AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                      AS critical_test_count,
  w.abnormal_rate_pct,
  w.critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                    AS prev_month_abnormal_rate_pct,
  CAST(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag w
LEFT JOIN {source_schema}.calendar c
  ON c.datekey = w.year * 10000 + w.month * 100 + 1
""")

In [0]:
updated_df = spark.table(target_table)
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))